# حاسوب زهراء الخارق ⚽
### موديل توقعات الدوري الإسباني — تحديث أسبوعي

**طريقة الاستخدام:** من القائمة اختاري **Runtime ← Run all** (أو Ctrl+F9). التشغيل ياخذ تقريباً دقيقة.

الموديل يسحب آخر النتائج والأودز من football-data.co.uk، يحسب قوة كل فريق (Dixon-Coles + توقعات السوق)، يحاكي باقي الموسم 10,000 مرة بقواعد الليغا الرسمية، ويطلع نسب اللقب، دوري الأبطال، والهبوط، وتوقعات الجولة الجاية.

**أفضل وقت للتشغيل:** الاثنين أو الثلاثاء بعد نهاية الجولة (الموقع يحدّث النتائج مرتين بالأسبوع)، أو الجمعة لتوقعات مباريات نهاية الأسبوع مع الأودز.

في آخر الدفتر ينزل ملف `dashboard_files.zip` — ابعثيه لـClaude حتى تتحدث صفحة النتائج.

In [ ]:
!pip -q install arabic-reshaper python-bidi
!mkdir -p zsc

In [ ]:
%%writefile zsc/__init__.py
"""Zahraa's Supercomputer (حاسوب زهراء الخارق) - a La Liga forecasting model.

Pipeline:
    data  ->  Dixon-Coles team ratings (goals + shots-on-target + market odds)
          ->  match probabilities  ->  Monte Carlo season simulation
          ->  title / Champions League / relegation probabilities
"""

__version__ = "1.0.0"


In [ ]:
%%writefile zsc/teams.py
"""Team names.

Canonical names follow football-data.co.uk (the main data source), so files
from that site can be used without any renaming. Every other source is mapped
onto these names through ALIASES.
"""

from __future__ import annotations

# canonical name -> (English display name, Arabic display name)
DISPLAY = {
    "Alaves": ("Alavés", "ألافيس"),
    "Almeria": ("Almería", "ألميريا"),
    "Ath Bilbao": ("Athletic Club", "أتلتيك بلباو"),
    "Ath Madrid": ("Atlético Madrid", "أتلتيكو مدريد"),
    "Barcelona": ("Barcelona", "برشلونة"),
    "Betis": ("Real Betis", "ريال بيتيس"),
    "Cadiz": ("Cádiz", "قادش"),
    "Celta": ("Celta Vigo", "سيلتا فيغو"),
    "Cordoba": ("Córdoba", "قرطبة"),
    "Eibar": ("Eibar", "إيبار"),
    "Elche": ("Elche", "إلتشي"),
    "Espanol": ("Espanyol", "إسبانيول"),
    "Getafe": ("Getafe", "خيتافي"),
    "Girona": ("Girona", "جيرونا"),
    "Granada": ("Granada", "غرناطة"),
    "Huesca": ("Huesca", "هويسكا"),
    "La Coruna": ("Deportivo La Coruña", "ديبورتيفو لاكورونيا"),
    "Las Palmas": ("Las Palmas", "لاس بالماس"),
    "Leganes": ("Leganés", "ليغانيس"),
    "Levante": ("Levante", "ليفانتي"),
    "Malaga": ("Málaga", "ملقة"),
    "Mallorca": ("Mallorca", "مايوركا"),
    "Osasuna": ("Osasuna", "أوساسونا"),
    "Oviedo": ("Real Oviedo", "ريال أوفييدو"),
    "Real Madrid": ("Real Madrid", "ريال مدريد"),
    "Santander": ("Racing Santander", "راسينغ سانتاندير"),
    "Sevilla": ("Sevilla", "إشبيلية"),
    "Sociedad": ("Real Sociedad", "ريال سوسيداد"),
    "Sp Gijon": ("Sporting Gijón", "سبورتينغ خيخون"),
    "Valencia": ("Valencia", "فالنسيا"),
    "Valladolid": ("Valladolid", "بلد الوليد"),
    "Vallecano": ("Rayo Vallecano", "رايو فاييكانو"),
    "Villarreal": ("Villarreal", "فياريال"),
    "Zaragoza": ("Zaragoza", "سرقسطة"),
}

# other spellings -> canonical name
ALIASES = {
    # football-data.co.uk historical spelling
    "Villareal": "Villarreal",
    # openfootball
    "Athletic Club": "Ath Bilbao",
    "CA Osasuna": "Osasuna",
    "Club Atlético de Madrid": "Ath Madrid",
    "Deportivo Alavés": "Alaves",
    "Elche CF": "Elche",
    "FC Barcelona": "Barcelona",
    "Getafe CF": "Getafe",
    "Levante UD": "Levante",
    "Málaga CF": "Malaga",
    "RC Celta de Vigo": "Celta",
    "RC Deportivo La Coruña": "La Coruna",
    "RCD Espanyol de Barcelona": "Espanol",
    "Rayo Vallecano de Madrid": "Vallecano",
    "Real Betis Balompié": "Betis",
    "Real Madrid CF": "Real Madrid",
    "Real Racing Club de Santander": "Santander",
    "Real Sociedad de Fútbol": "Sociedad",
    "Sevilla FC": "Sevilla",
    "Valencia CF": "Valencia",
    "Villarreal CF": "Villarreal",
    "Girona FC": "Girona",
    "RCD Mallorca": "Mallorca",
    "Real Oviedo": "Oviedo",
    # footballwebpages.co.uk / common English names
    "Athletic Bilbao": "Ath Bilbao",
    "Atlético Madrid": "Ath Madrid",
    "Atletico Madrid": "Ath Madrid",
    "Alavés": "Alaves",
    "Real Betis": "Betis",
    "Real Sociedad": "Sociedad",
    "Celta Vigo": "Celta",
    "Deportivo de A Coruña": "La Coruna",
    "Deportivo La Coruña": "La Coruna",
    "Deportivo La Coruna": "La Coruna",
    "Espanyol": "Espanol",
    "Rayo Vallecano": "Vallecano",
    "Racing de Santander": "Santander",
    "Racing Santander": "Santander",
    "Málaga": "Malaga",
    # SportRadar
    "Real Betis Seville": "Betis",
    "Malaga CF": "Malaga",
    "Real Sociedad San Sebastian": "Sociedad",
    "Espanyol Barcelona": "Espanol",
    "RC Deportivo de A Coruna": "La Coruna",
    "Atletico Madrid": "Ath Madrid",
    "Deportivo Alaves": "Alaves",
    "CA Osasuna": "Osasuna",
    # StatsBomb
    "Barcelona": "Barcelona",
    "Real Madrid": "Real Madrid",
    "Atlético Madrid": "Ath Madrid",
    "RC Deportivo La Coruña": "La Coruna",
    "Real Valladolid": "Valladolid",
    "Sporting Gijón": "Sp Gijon",
    "Granada": "Granada",
    "Eibar": "Eibar",
    "Las Palmas": "Las Palmas",
}


def canonical(name: str) -> str:
    """Return the canonical (football-data.co.uk) name for any spelling."""
    name = str(name).strip()
    if name in DISPLAY:
        return name
    return ALIASES.get(name, name)


def english(name: str) -> str:
    return DISPLAY.get(canonical(name), (name, name))[0]


def arabic(name: str) -> str:
    return DISPLAY.get(canonical(name), (name, name))[1]


In [ ]:
%%writefile zsc/data.py
"""Loading and validating match data.

Unified match table columns (one row per played match):
    date      pandas Timestamp (match day)
    season    int, first year of the season (2026 = 2026/27)
    home, away  canonical team names (see teams.py)
    hg, ag    full-time goals
    hst, ast  shots on target (NaN if unknown)
    odds_h, odds_d, odds_a   1X2 decimal odds (NaN if unknown)
    odds_o25, odds_u25       over/under 2.5 goals odds (NaN if unknown)
    hthg, htag               half-time goals (NaN if unknown)
"""

from __future__ import annotations

import glob
import io
import os
import re
from itertools import permutations

import numpy as np
import pandas as pd

from .teams import canonical

COLUMNS = ["date", "season", "home", "away", "hg", "ag", "hst", "ast",
           "odds_h", "odds_d", "odds_a", "odds_o25", "odds_u25", "hthg", "htag"]

FOOTBALL_DATA_URL = "https://www.football-data.co.uk/mmz4281/{code}/SP1.csv"


def season_code(start_year: int) -> str:
    """2026 -> '2627' (football-data.co.uk folder name)."""
    return f"{start_year % 100:02d}{(start_year + 1) % 100:02d}"


def code_to_start_year(code: str) -> int:
    first = int(code[:2])
    return 1900 + first if first >= 90 else 2000 + first


def _parse_dates(s: pd.Series) -> pd.Series:
    s = s.astype(str).str.strip()
    if s.str.match(r"^\d{4}-\d{2}-\d{2}").all():
        return pd.to_datetime(s.str[:10], format="%Y-%m-%d")
    # football-data.co.uk uses dd/mm/yy and dd/mm/yyyy
    out = pd.to_datetime(s, format="%d/%m/%Y", errors="coerce")
    short = out.isna()
    if short.any():
        out[short] = pd.to_datetime(s[short], format="%d/%m/%y", errors="coerce")
    if out.isna().any():
        bad = s[out.isna()].unique()[:5]
        raise ValueError(f"Unparseable dates: {bad}")
    return out


def _first_present(df: pd.DataFrame, names: list[str]) -> pd.Series:
    for n in names:
        if n in df.columns:
            return pd.to_numeric(df[n], errors="coerce")
    return pd.Series(np.nan, index=df.index)


def standardise_football_data(raw: pd.DataFrame, season: int) -> pd.DataFrame:
    """Convert a football-data.co.uk style table to the unified format."""
    raw = raw.dropna(subset=["HomeTeam", "AwayTeam", "FTHG", "FTAG"]).copy()
    out = pd.DataFrame({
        "date": _parse_dates(raw["Date"]),
        "season": season,
        "home": raw["HomeTeam"].map(canonical),
        "away": raw["AwayTeam"].map(canonical),
        "hg": pd.to_numeric(raw["FTHG"]).astype(int),
        "ag": pd.to_numeric(raw["FTAG"]).astype(int),
        "hst": _first_present(raw, ["HST"]),
        "ast": _first_present(raw, ["AST"]),
        # Bet365 first (the backtests were tuned on Bet365), market average next
        "odds_h": _first_present(raw, ["B365H", "AvgH", "BbAvH"]),
        "odds_d": _first_present(raw, ["B365D", "AvgD", "BbAvD"]),
        "odds_a": _first_present(raw, ["B365A", "AvgA", "BbAvA"]),
        "odds_o25": _first_present(raw, ["B365>2.5", "Avg>2.5", "BbAv>2.5"]),
        "odds_u25": _first_present(raw, ["B365<2.5", "Avg<2.5", "BbAv<2.5"]),
        "hthg": _first_present(raw, ["HTHG"]),
        "htag": _first_present(raw, ["HTAG"]),
    })
    return out[COLUMNS].reset_index(drop=True)


def load_history(history_dir: str, first_season: int = 1993) -> pd.DataFrame:
    """Read season-XXYY.csv files (football-data.co.uk columns)."""
    frames = []
    for path in sorted(glob.glob(os.path.join(history_dir, "season-*.csv"))):
        code = re.search(r"season-(\d{4})\.csv$", path).group(1)
        start = code_to_start_year(code)
        if start < first_season:
            continue
        frames.append(standardise_football_data(pd.read_csv(path), start))
    if not frames:
        raise FileNotFoundError(f"No season files in {history_dir}")
    df = pd.concat(frames, ignore_index=True)
    return df.sort_values(["date", "home"]).reset_index(drop=True)


def attach_odds(matches: pd.DataFrame, odds: pd.DataFrame) -> pd.DataFrame:
    """Fill missing odds / shots columns from a second table.

    The second table must use the unified column names. Matches are joined on
    (date, home, away); goals must agree, otherwise a ValueError is raised.
    """
    key = ["date", "home", "away"]
    o = odds.copy()
    o["home"] = o["home"].map(canonical)
    o["away"] = o["away"].map(canonical)
    merged = matches.merge(o, on=key, how="left", suffixes=("", "_o"))
    both = merged["hg_o"].notna()
    bad = both & ((merged["hg"] != merged["hg_o"]) | (merged["ag"] != merged["ag_o"]))
    if bad.any():
        raise ValueError("Score mismatch between sources:\n"
                         + merged.loc[bad, key + ["hg", "ag", "hg_o", "ag_o"]].to_string())
    for c in ["hst", "ast", "odds_h", "odds_d", "odds_a", "odds_o25", "odds_u25", "hthg", "htag"]:
        if c + "_o" in merged:
            merged[c] = merged[c].fillna(merged[c + "_o"])
    return merged[COLUMNS]


def load_unified_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["date"] = _parse_dates(df["date"])
    df["home"] = df["home"].map(canonical)
    df["away"] = df["away"].map(canonical)
    for c in COLUMNS:
        if c not in df:
            df[c] = np.nan
    df["hg"] = df["hg"].astype(int)
    df["ag"] = df["ag"].astype(int)
    df["season"] = df["season"].astype(int)
    return df[COLUMNS].sort_values(["date", "home"]).reset_index(drop=True)


def fetch_football_data(start_year: int, url_template: str = FOOTBALL_DATA_URL) -> pd.DataFrame:
    """Download one season from football-data.co.uk (needs internet)."""
    import urllib.request
    url = url_template.format(code=season_code(start_year))
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (zsc model)"})
    with urllib.request.urlopen(req, timeout=60) as r:
        text = r.read().decode("utf-8-sig", errors="replace")
    return standardise_football_data(pd.read_csv(io.StringIO(text)), start_year)


# --------------------------------------------------------------------------
# validation
# --------------------------------------------------------------------------

def validate_season(df: pd.DataFrame, n_teams: int | None = None, complete: bool = False) -> list[str]:
    """Return a list of problems (empty list = clean).

    Checks: no duplicate fixtures, no team playing itself, goals are
    non-negative integers, each ordered pair at most once, and (if complete)
    a full double round robin.
    """
    problems = []
    teams = sorted(set(df["home"]) | set(df["away"]))
    if n_teams is not None and len(teams) != n_teams:
        problems.append(f"expected {n_teams} teams, found {len(teams)}: {teams}")
    if (df["home"] == df["away"]).any():
        problems.append("a team plays itself")
    dup = df.duplicated(["home", "away"], keep=False)
    if dup.any():
        problems.append("duplicate fixtures: " + df.loc[dup, ["date", "home", "away"]].to_string())
    if (df["hg"] < 0).any() or (df["ag"] < 0).any():
        problems.append("negative goals")
    if complete:
        n = len(teams)
        if len(df) != n * (n - 1):
            problems.append(f"expected {n * (n - 1)} matches, found {len(df)}")
    return problems


def remaining_fixtures(played: pd.DataFrame, teams: list[str]) -> pd.DataFrame:
    """All (home, away) pairs of a double round robin that are not yet played."""
    done = set(zip(played["home"], played["away"]))
    rows = [(h, a) for h, a in permutations(sorted(teams), 2) if (h, a) not in done]
    return pd.DataFrame(rows, columns=["home", "away"])


In [ ]:
%%writefile zsc/dixon_coles.py
"""Dixon-Coles team-strength model.

    goals_home ~ Poisson(lambda),  lambda = exp(c + h + attack[home] - defence[away])
    goals_away ~ Poisson(mu),      mu     = exp(c +     attack[away] - defence[home])

with the Dixon & Coles (1997) correction tau(x, y) for the low scores
0-0, 1-0, 0-1, 1-1, exponential time-decay weights, and a normal prior on
every team rating (promoted teams get a lower prior mean).

The "target" the Poisson part is fitted to can blend three signals of team
strength for each past match:
    * the real goals
    * a shots-on-target proxy for expected goals  (k * shots on target)
    * the goals the betting market expected before kick-off
The tau correction always uses the real score.
"""

from __future__ import annotations

from dataclasses import dataclass, field

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import poisson

MAX_GOALS = 10
RHO_BOUNDS = (-0.25, 0.10)


# --------------------------------------------------------------------------
# score distributions
# --------------------------------------------------------------------------

def score_matrix(lam, mu, rho, max_goals: int = MAX_GOALS) -> np.ndarray:
    """P(home = x, away = y) for each match; shape (n, G+1, G+1)."""
    lam = np.atleast_1d(np.asarray(lam, dtype=float))
    mu = np.atleast_1d(np.asarray(mu, dtype=float))
    rho = np.broadcast_to(np.asarray(rho, dtype=float), lam.shape)
    k = np.arange(max_goals + 1)
    ph = poisson.pmf(k[None, :], lam[:, None])
    pa = poisson.pmf(k[None, :], mu[:, None])
    m = ph[:, :, None] * pa[:, None, :]
    m[:, 0, 0] *= 1.0 - lam * mu * rho
    m[:, 0, 1] *= 1.0 + lam * rho
    m[:, 1, 0] *= 1.0 + mu * rho
    m[:, 1, 1] *= 1.0 - rho
    m = np.clip(m, 0.0, None)
    return m / m.sum(axis=(1, 2), keepdims=True)


def outcome_probs(m: np.ndarray) -> np.ndarray:
    """(n, 3) array of P(home win), P(draw), P(away win)."""
    g = m.shape[1]
    home = np.tril(np.ones((g, g), dtype=bool), -1)      # x > y
    away = np.triu(np.ones((g, g), dtype=bool), 1)       # x < y
    draw = np.eye(g, dtype=bool)
    return np.stack([m[:, home].sum(1), m[:, draw].sum(1), m[:, away].sum(1)], axis=1)


def over_prob(m: np.ndarray, line: float = 2.5) -> np.ndarray:
    g = m.shape[1]
    tot = np.add.outer(np.arange(g), np.arange(g))
    return m[:, tot > line].sum(1)


# --------------------------------------------------------------------------
# model
# --------------------------------------------------------------------------

@dataclass
class DCConfig:
    xi: float = 0.0019              # time decay per day (weight = exp(-xi * days))
    prior_sd: float = 0.35          # sd of the normal prior on attack/defence
    beta_sot: float = 0.0           # weight of the shots-on-target xG proxy in the target
    beta_mkt: float = 0.0           # weight of market-implied goals in the target
    promoted_attack: float = -0.19  # prior mean for newly promoted teams
    promoted_defence: float = -0.13  # (average first season of promoted teams, 2003-2018)
    seasons_back: int = 4           # seasons of history before the current one
    sot_k: float | None = None      # goals per shot on target (estimated if None)
    rating_scale: float = 1.0       # stretch of rating differences at prediction time
                                    # (>1 corrects the shrinkage of time-averaged ratings)


@dataclass
class FittedDC:
    teams: list
    c: float
    h: float
    rho: float
    attack: np.ndarray
    defence: np.ndarray
    cov: np.ndarray | None = None          # Laplace covariance of the full parameter vector
    info: dict = field(default_factory=dict)
    scale: float = 1.0                     # rating_scale
    center_a: float = 0.0                  # mean attack / defence of the current teams
    center_d: float = 0.0

    @property
    def index(self):
        return {t: i for i, t in enumerate(self.teams)}

    def theta(self) -> np.ndarray:
        return np.concatenate([[self.c, self.h, self.rho], self.attack, self.defence])

    def scaled(self, theta: np.ndarray | None = None):
        """Attack and defence after applying rating_scale around the current-team mean."""
        th = self.theta() if theta is None else theta
        n = len(self.teams)
        att, dfn = th[3:3 + n], th[3 + n:]
        return (self.center_a + self.scale * (att - self.center_a),
                self.center_d + self.scale * (dfn - self.center_d))

    def rates(self, home, away, theta: np.ndarray | None = None):
        """Expected goals (lambda, mu) for arrays of fixtures."""
        idx = self.index
        hi = np.array([idx[t] for t in home], dtype=int)
        ai = np.array([idx[t] for t in away], dtype=int)
        th = self.theta() if theta is None else theta
        c, h = th[0], th[1]
        att, dfn = self.scaled(th)
        lam = np.exp(c + h + att[hi] - dfn[ai])
        mu = np.exp(c + att[ai] - dfn[hi])
        return lam, mu

    def predict(self, home, away) -> pd.DataFrame:
        home, away = list(home), list(away)
        if not home:
            return pd.DataFrame(columns=["home", "away", "xg_home", "xg_away", "p_home", "p_draw",
                                         "p_away", "likely_score", "p_likely_score"])
        lam, mu = self.rates(home, away)
        m = score_matrix(lam, mu, self.rho)
        p = outcome_probs(m)
        best = m.reshape(len(home), -1).argmax(1)
        g = m.shape[1]
        return pd.DataFrame({
            "home": home, "away": away, "xg_home": lam, "xg_away": mu,
            "p_home": p[:, 0], "p_draw": p[:, 1], "p_away": p[:, 2],
            "likely_score": [f"{b // g}-{b % g}" for b in best],
            "p_likely_score": m.reshape(len(home), -1).max(1),
        })

    def sample_theta(self, n: int, rng: np.random.Generator) -> np.ndarray:
        """Parameter draws from the Laplace approximation (n, p)."""
        th = self.theta()
        if self.cov is None:
            return np.repeat(th[None, :], n, axis=0)
        draws = rng.multivariate_normal(th, self.cov, size=n, method="cholesky")
        draws[:, 2] = np.clip(draws[:, 2], *RHO_BOUNDS)
        return draws


def _blend_targets(df: pd.DataFrame, cfg: DCConfig, k: float):
    """Per-match Poisson targets (home, away) mixing goals / SOT proxy / market."""
    n = len(df)
    wg = np.full(n, max(0.0, 1.0 - cfg.beta_sot - cfg.beta_mkt))
    has_sot = df["hst"].notna().values & df["ast"].notna().values
    has_mkt = df["mkt_lh"].notna().values & df["mkt_la"].notna().values if "mkt_lh" in df else np.zeros(n, bool)
    ws = np.where(has_sot, cfg.beta_sot, 0.0)
    wm = np.where(has_mkt, cfg.beta_mkt, 0.0)
    tot = wg + ws + wm
    tot = np.where(tot <= 0, 1.0, tot)
    wg = np.where(wg + ws + wm <= 0, 1.0, wg)
    hg, ag = df["hg"].values.astype(float), df["ag"].values.astype(float)
    sh = np.nan_to_num(df["hst"].values.astype(float)) * k
    sa = np.nan_to_num(df["ast"].values.astype(float)) * k
    if "mkt_lh" in df:
        mh, ma = np.nan_to_num(df["mkt_lh"].values.astype(float)), np.nan_to_num(df["mkt_la"].values.astype(float))
    else:
        mh = ma = np.zeros(n)
    th = (wg * hg + ws * sh + wm * mh) / tot
    ta = (wg * ag + ws * sa + wm * ma) / tot
    return th, ta


class _Objective:
    """Negative penalised log-likelihood and its exact gradient."""

    def __init__(self, hi, ai, x, y, th, ta, w, prior_a, prior_d, prior_sd, n_teams):
        self.hi, self.ai, self.x, self.y = hi, ai, x, y
        self.th, self.ta, self.w = th, ta, w
        self.pa, self.pd = prior_a, prior_d
        self.inv_var = 1.0 / prior_sd ** 2
        self.n = n_teams
        self.m00 = (x == 0) & (y == 0)
        self.m01 = (x == 0) & (y == 1)
        self.m10 = (x == 1) & (y == 0)
        self.m11 = (x == 1) & (y == 1)

    def __call__(self, theta):
        n = self.n
        c, h, rho = theta[0], theta[1], theta[2]
        att, dfn = theta[3:3 + n], theta[3 + n:]
        loglam = c + h + att[self.hi] - dfn[self.ai]
        logmu = c + att[self.ai] - dfn[self.hi]
        lam, mu = np.exp(loglam), np.exp(logmu)

        tau = np.ones_like(lam)
        tau[self.m00] = 1.0 - lam[self.m00] * mu[self.m00] * rho
        tau[self.m01] = 1.0 + lam[self.m01] * rho
        tau[self.m10] = 1.0 + mu[self.m10] * rho
        tau[self.m11] = 1.0 - rho
        tau = np.maximum(tau, 1e-10)

        ll = self.w * (self.th * loglam - lam + self.ta * logmu - mu + np.log(tau))
        pen = 0.5 * self.inv_var * (np.sum((att - self.pa) ** 2) + np.sum((dfn - self.pd) ** 2))
        f = -ll.sum() + pen

        # d log tau / d log lambda, d log mu, d rho
        dl = np.zeros_like(lam)
        dm = np.zeros_like(lam)
        dr = np.zeros_like(lam)
        s = self.m00
        dl[s] = -lam[s] * mu[s] * rho / tau[s]
        dm[s] = dl[s]
        dr[s] = -lam[s] * mu[s] / tau[s]
        s = self.m01
        dl[s] = lam[s] * rho / tau[s]
        dr[s] = lam[s] / tau[s]
        s = self.m10
        dm[s] = mu[s] * rho / tau[s]
        dr[s] = mu[s] / tau[s]
        s = self.m11
        dr[s] = -1.0 / tau[s]

        g_l = self.w * (self.th - lam + dl)      # d ll / d log lambda
        g_m = self.w * (self.ta - mu + dm)       # d ll / d log mu
        g_r = self.w * dr

        grad = np.empty_like(theta)
        grad[0] = -(g_l.sum() + g_m.sum())
        grad[1] = -g_l.sum()
        grad[2] = -g_r.sum()
        ga = np.bincount(self.hi, g_l, n) + np.bincount(self.ai, g_m, n)
        gd = np.bincount(self.ai, g_l, n) + np.bincount(self.hi, g_m, n)
        grad[3:3 + n] = -ga + self.inv_var * (att - self.pa)
        grad[3 + n:] = gd + self.inv_var * (dfn - self.pd)
        return f, grad


def fit_dixon_coles(matches: pd.DataFrame, ref_date, cfg: DCConfig,
                    current_teams: list[str] | None = None,
                    promoted: set[str] | None = None,
                    current_season: int | None = None,
                    x0: FittedDC | None = None,
                    laplace: bool = True) -> FittedDC:
    """Fit on matches played strictly before ref_date.

    current_teams  teams of the season being forecast (kept even with no data)
    promoted       teams that get the promoted-team prior mean
    """
    ref_date = pd.Timestamp(ref_date)
    df = matches[matches["date"] < ref_date]
    if current_season is not None:
        df = df[df["season"] >= current_season - cfg.seasons_back]
    df = df.reset_index(drop=True)

    teams = sorted(set(df["home"]) | set(df["away"]) | set(current_teams or []))
    idx = {t: i for i, t in enumerate(teams)}
    n = len(teams)
    hi = df["home"].map(idx).values
    ai = df["away"].map(idx).values
    x = df["hg"].values.astype(int)
    y = df["ag"].values.astype(int)
    days = (ref_date - df["date"]).dt.days.values.astype(float)
    w = np.exp(-cfg.xi * days)

    if cfg.sot_k is not None:
        k = cfg.sot_k
    else:
        ok = df["hst"].notna() & df["ast"].notna()
        sot = (df.loc[ok, "hst"] + df.loc[ok, "ast"]).sum()
        k = float((df.loc[ok, "hg"] + df.loc[ok, "ag"]).sum() / sot) if sot > 0 else 0.0
    th, ta = _blend_targets(df, cfg, k)

    promoted = promoted or set()
    prior_a = np.array([cfg.promoted_attack if t in promoted else 0.0 for t in teams])
    prior_d = np.array([cfg.promoted_defence if t in promoted else 0.0 for t in teams])
    obj = _Objective(hi, ai, x, y, th, ta, w, prior_a, prior_d, cfg.prior_sd, n)

    if x0 is not None:
        old = x0.index
        att0 = np.array([x0.attack[old[t]] if t in old else prior_a[i] for i, t in enumerate(teams)])
        def0 = np.array([x0.defence[old[t]] if t in old else prior_d[i] for i, t in enumerate(teams)])
        start = np.concatenate([[x0.c, x0.h, x0.rho], att0, def0])
    else:
        mean_goals = max(0.5, (np.sum(w * (x + y)) / (2 * np.sum(w))) if len(df) else 1.3)
        start = np.concatenate([[np.log(mean_goals), 0.2, -0.05], prior_a, prior_d])

    bounds = [(None, None), (None, None), RHO_BOUNDS] + [(None, None)] * (2 * n)
    res = minimize(obj, start, jac=True, method="L-BFGS-B", bounds=bounds,
                   options={"maxiter": 2000, "ftol": 1e-12, "gtol": 1e-7})
    theta = res.x
    centre_idx = [idx[t] for t in (current_teams or teams)]
    fit = FittedDC(teams=teams, c=theta[0], h=theta[1], rho=theta[2],
                   attack=theta[3:3 + n].copy(), defence=theta[3 + n:].copy(),
                   info={"converged": bool(res.success), "message": str(res.message),
                         "n_matches": int(len(df)), "sot_k": k, "nll": float(res.fun),
                         "eff_matches": float(w.sum())},
                   scale=cfg.rating_scale,
                   center_a=float(theta[3:3 + n][centre_idx].mean()),
                   center_d=float(theta[3 + n:][centre_idx].mean()))
    if laplace:
        fit.cov = _laplace_cov(obj, theta)
    return fit


def _laplace_cov(obj: _Objective, theta: np.ndarray, eps: float = 1e-5) -> np.ndarray:
    """Inverse Hessian at the optimum (numerical differentiation of the exact gradient)."""
    p = len(theta)
    hess = np.empty((p, p))
    for i in range(p):
        e = np.zeros(p)
        e[i] = eps
        hess[:, i] = (obj(theta + e)[1] - obj(theta - e)[1]) / (2 * eps)
    hess = 0.5 * (hess + hess.T)
    # tiny ridge for numerical safety, then invert
    vals, vecs = np.linalg.eigh(hess)
    vals = np.maximum(vals, 1e-8)
    return (vecs / vals) @ vecs.T


In [ ]:
%%writefile zsc/market.py
"""Turning bookmaker odds into probabilities and expected goals."""

from __future__ import annotations

import numpy as np
import pandas as pd

from .dixon_coles import outcome_probs, over_prob, score_matrix


def implied_1x2(odds_h, odds_d, odds_a) -> np.ndarray:
    """Margin-free probabilities (proportional method). Shape (n, 3)."""
    inv = 1.0 / np.column_stack([odds_h, odds_d, odds_a]).astype(float)
    return inv / inv.sum(axis=1, keepdims=True)


def implied_over(odds_over, odds_under) -> np.ndarray:
    inv_o = 1.0 / np.asarray(odds_over, dtype=float)
    inv_u = 1.0 / np.asarray(odds_under, dtype=float)
    return inv_o / (inv_o + inv_u)


def _model_probs(u, v, rho):
    m = score_matrix(np.exp(u), np.exp(v), rho)
    p = outcome_probs(m)
    return p[:, 0], p[:, 2], over_prob(m)


def implied_goals(p_home, p_away, p_over=None, rho: float = -0.05,
                  iters: int = 40, tol: float = 1e-10):
    """Solve for (lambda, mu) whose Dixon-Coles probabilities match the market.

    Least squares on P(home), P(away) and, when given, P(over 2.5).
    Vectorised Levenberg-Marquardt; returns (lambda, mu, max_abs_residual).
    """
    p_home = np.asarray(p_home, float)
    p_away = np.asarray(p_away, float)
    n = len(p_home)
    has_o = np.zeros(n, bool) if p_over is None else ~np.isnan(np.asarray(p_over, float))
    p_over = np.zeros(n) if p_over is None else np.nan_to_num(np.asarray(p_over, float))

    # starting point from the odds ratio
    u = np.log(1.35) + 0.6 * (p_home - p_away)
    v = np.log(1.10) - 0.6 * (p_home - p_away)
    damp = np.full(n, 1e-3)
    eps = 1e-5

    def resid(u_, v_):
        ph, pa, po = _model_probs(u_, v_, rho)
        r = np.stack([ph - p_home, pa - p_away, np.where(has_o, po - p_over, 0.0)], axis=1)
        return r

    r = resid(u, v)
    for _ in range(iters):
        ru = (resid(u + eps, v) - resid(u - eps, v)) / (2 * eps)
        rv = (resid(u, v + eps) - resid(u, v - eps)) / (2 * eps)
        # normal equations per match (2x2)
        a11 = (ru * ru).sum(1) + damp
        a22 = (rv * rv).sum(1) + damp
        a12 = (ru * rv).sum(1)
        b1 = -(ru * r).sum(1)
        b2 = -(rv * r).sum(1)
        det = a11 * a22 - a12 ** 2
        du = (a22 * b1 - a12 * b2) / det
        dv = (a11 * b2 - a12 * b1) / det
        du = np.clip(du, -0.5, 0.5)
        dv = np.clip(dv, -0.5, 0.5)
        r_new = resid(u + du, v + dv)
        better = (r_new ** 2).sum(1) <= (r ** 2).sum(1)
        u = np.where(better, u + du, u)
        v = np.where(better, v + dv, v)
        r = np.where(better[:, None], r_new, r)
        damp = np.where(better, damp * 0.3, damp * 10)
        if np.abs(r).max() < tol:
            break
    return np.exp(u), np.exp(v), np.abs(r).max(axis=1)


def add_market_goals(matches: pd.DataFrame, rho: float = -0.05) -> pd.DataFrame:
    """Add market-implied expected goals (mkt_lh, mkt_la) where odds exist."""
    out = matches.copy()
    out["mkt_lh"] = np.nan
    out["mkt_la"] = np.nan
    odds = out[["odds_h", "odds_d", "odds_a"]]
    ok = np.asarray(odds.notna().all(axis=1)) & np.asarray((odds.fillna(0) > 1.0).all(axis=1))
    if not ok.any():
        return out
    sub = out.loc[ok]
    p = implied_1x2(sub["odds_h"], sub["odds_d"], sub["odds_a"])
    po = np.full(len(sub), np.nan)
    has_ou = sub[["odds_o25", "odds_u25"]].notna().all(axis=1).values
    po[has_ou] = implied_over(sub.loc[has_ou, "odds_o25"], sub.loc[has_ou, "odds_u25"])
    lam, mu, err = implied_goals(p[:, 0], p[:, 2], po, rho=rho)
    out.loc[ok, "mkt_lh"] = lam
    out.loc[ok, "mkt_la"] = mu
    out.loc[ok, "mkt_err"] = err
    return out


In [ ]:
%%writefile zsc/standings.py
"""League tables and the official La Liga tie-breaking rules.

Teams level on points are separated as follows (RFEF / LaLiga rules):

  If every tied team has played every other tied team twice:
    * 2 teams      -> head-to-head goal difference (no away-goals rule)
    * 3+ teams     -> head-to-head points, then head-to-head goal difference,
                      using only the matches between the tied teams.
                      Teams still tied are re-ranked from the start among
                      themselves.
  Otherwise, or if still tied:
    * overall goal difference, then overall goals scored
    * then Fair Play, then a play-off match. Neither can be predicted, so the
      simulator breaks such ties at random and the table display by name.
"""

from __future__ import annotations

import numpy as np
import pandas as pd


def compute_table(matches: pd.DataFrame, teams: list[str] | None = None) -> pd.DataFrame:
    """Points table (unordered) with P W D L GF GA GD Pts."""
    if teams is None:
        teams = sorted(set(matches["home"]) | set(matches["away"]))
    t = pd.DataFrame(0, index=teams, columns=["P", "W", "D", "L", "GF", "GA"])
    for h, a, x, y in zip(matches["home"], matches["away"], matches["hg"], matches["ag"]):
        t.loc[h, ["P", "GF", "GA"]] += [1, x, y]
        t.loc[a, ["P", "GF", "GA"]] += [1, y, x]
        if x > y:
            t.loc[h, "W"] += 1
            t.loc[a, "L"] += 1
        elif x < y:
            t.loc[a, "W"] += 1
            t.loc[h, "L"] += 1
        else:
            t.loc[h, "D"] += 1
            t.loc[a, "D"] += 1
    t["GD"] = t["GF"] - t["GA"]
    t["Pts"] = 3 * t["W"] + t["D"]
    return t.rename_axis("team").reset_index()


def h2h_matrices(matches: pd.DataFrame, teams: list[str]):
    """pts[i, j] = points i took from j, gf[i, j] = goals i scored v j,
    played[i, j] = meetings played."""
    idx = {t: k for k, t in enumerate(teams)}
    n = len(teams)
    pts = np.zeros((n, n), dtype=np.int64)
    gf = np.zeros((n, n), dtype=np.int64)
    played = np.zeros((n, n), dtype=np.int64)
    for h, a, x, y in zip(matches["home"], matches["away"], matches["hg"], matches["ag"]):
        i, j = idx[h], idx[a]
        gf[i, j] += x
        gf[j, i] += y
        played[i, j] += 1
        played[j, i] += 1
        if x > y:
            pts[i, j] += 3
        elif x < y:
            pts[j, i] += 3
        else:
            pts[i, j] += 1
            pts[j, i] += 1
    return pts, gf, played


def _split(group: list[int], values: np.ndarray) -> list[list[int]]:
    """Split a group into runs of equal value, best (highest) first."""
    order = sorted(group, key=lambda k: -values[k])
    parts, cur = [], [order[0]]
    for k in order[1:]:
        if values[k] == values[cur[-1]]:
            cur.append(k)
        else:
            parts.append(cur)
            cur = [k]
    parts.append(cur)
    return parts


class _Resolver:
    def __init__(self, gd, gf, h2h_pts, h2h_gf, h2h_played, final_key):
        self.gd, self.gf = gd, gf
        self.h2h_pts, self.h2h_gf, self.h2h_played = h2h_pts, h2h_gf, h2h_played
        self.final_key = final_key          # array: last resort ordering value

    def h2h_complete(self, g):
        sub = self.h2h_played[np.ix_(g, g)]
        off = ~np.eye(len(g), dtype=bool)
        return bool((sub[off] >= 2).all())

    def resolve(self, g: list[int]) -> list[int]:
        if len(g) == 1:
            return list(g)
        if self.h2h_complete(g):
            n = len(self.gd)
            if len(g) == 2:
                i, j = g
                d = self.h2h_gf[i, j] - self.h2h_gf[j, i]
                if d != 0:
                    return [i, j] if d > 0 else [j, i]
            else:
                ix = np.ix_(g, g)
                mini_pts = np.zeros(n)
                mini_gd = np.zeros(n)
                mini_pts[g] = self.h2h_pts[ix].sum(axis=1)
                mini_gd[g] = self.h2h_gf[ix].sum(axis=1) - self.h2h_gf[ix].sum(axis=0)
                for vals in (mini_pts, mini_gd):
                    parts = _split(g, vals)
                    if len(parts) > 1:
                        out = []
                        for p in parts:           # still-tied teams start again
                            out.extend(self.resolve(p))
                        return out
        return self.overall(g)

    def overall(self, g: list[int]) -> list[int]:
        for vals in (self.gd, self.gf, self.final_key):
            parts = _split(g, vals)
            if len(parts) > 1:
                out = []
                for p in parts:
                    out.extend(self.overall(p) if len(p) > 1 else p)
                return out
        return list(g)       # identical on every key (cannot happen with a strict final key)


def rank_order(pts, gd, gf, h2h_pts, h2h_gf, h2h_played, final_key) -> list[int]:
    """Indices of teams from 1st to last.

    final_key must be strictly ordered (e.g. random numbers, or a reversed
    alphabetical rank) - it stands in for Fair Play.
    """
    pts = np.asarray(pts)
    res = _Resolver(np.asarray(gd), np.asarray(gf), h2h_pts, h2h_gf, h2h_played, np.asarray(final_key))
    order = []
    for grp in _split(list(range(len(pts))), pts):
        order.extend(res.resolve(grp))
    return order


def league_table(matches: pd.DataFrame, teams: list[str] | None = None) -> pd.DataFrame:
    """Ordered table using the official rules (ties beyond goals: by name)."""
    t = compute_table(matches, teams)
    teams = t["team"].tolist()
    hp, hg, hpl = h2h_matrices(matches, teams)
    # alphabetical final key: earlier name = higher value
    name_rank = np.argsort(np.argsort(teams))
    final_key = -name_rank.astype(float)
    order = rank_order(t["Pts"].values, t["GD"].values, t["GF"].values, hp, hg, hpl, final_key)
    out = t.iloc[order].reset_index(drop=True)
    out.insert(0, "Pos", np.arange(1, len(out) + 1))
    return out


In [ ]:
%%writefile zsc/simulate.py
"""Monte Carlo simulation of the rest of a season."""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd

from .data import remaining_fixtures
from .dixon_coles import FittedDC, score_matrix
from .standings import compute_table, h2h_matrices, rank_order


@dataclass
class SimResult:
    teams: list
    positions: np.ndarray      # (n_sims, n_teams) final position, 1 = champion
    points: np.ndarray         # (n_sims, n_teams) final points
    fixtures: pd.DataFrame     # remaining fixtures with model probabilities
    base_table: pd.DataFrame

    def summary(self, relegated: int = 3, ucl: int = 4, europe: int = 6) -> pd.DataFrame:
        n = len(self.teams)
        pos = self.positions
        out = pd.DataFrame({
            "team": self.teams,
            "pts_now": self.base_table.set_index("team").loc[self.teams, "Pts"].values,
            "exp_pts": self.points.mean(0),
            "pts_p10": np.percentile(self.points, 10, axis=0),
            "pts_p90": np.percentile(self.points, 90, axis=0),
            "exp_pos": pos.mean(0),
            "p_title": (pos == 1).mean(0),
            "p_top4": (pos <= ucl).mean(0),
            "p_top6": (pos <= europe).mean(0),
            "p_releg": (pos > n - relegated).mean(0),
        })
        for k in range(1, n + 1):
            out[f"pos_{k}"] = (pos == k).mean(0)
        return out.sort_values(["exp_pts", "p_title"], ascending=False).reset_index(drop=True)


def simulate_season(fit: FittedDC, played: pd.DataFrame, teams: list[str],
                    n_sims: int = 10_000, n_param_draws: int = 1000,
                    parameter_uncertainty: bool = True, drift_sd: float = 0.0,
                    overrides: pd.DataFrame | None = None, fixed_dist: dict | None = None,
                    seed: int = 2026) -> SimResult:
    """Simulate the remaining fixtures n_sims times.

    parameter_uncertainty  draw team ratings from the fitted model's
                           uncertainty (Laplace approximation)
    drift_sd               extra random change of every team's attack and
                           defence over the rest of the season (form, injuries,
                           transfers, coach changes), scaled by the square root
                           of the share of the season still to play
    overrides              optional table (home, away, mkt_lh, mkt_la): for these
                           fixtures use the betting market's expected goals
                           (the market beats any rating model on the very next
                           match because it knows team news)
    fixed_dist             optional {(home, away): matrix} of final-score
                           probabilities (rows = home goals) for matches in
                           progress, e.g. from the in-play model
    """
    rng = np.random.default_rng(seed)
    teams = sorted(teams)
    idx = {t: i for i, t in enumerate(teams)}
    n = len(teams)

    base = compute_table(played, teams)
    pts0 = base.set_index("team").loc[teams, "Pts"].values
    gf0 = base.set_index("team").loc[teams, "GF"].values
    ga0 = base.set_index("team").loc[teams, "GA"].values
    hp0, hg0, _ = h2h_matrices(played, teams)

    rem = remaining_fixtures(played, teams)
    rh = rem["home"].map(idx).values
    ra = rem["away"].map(idx).values
    R = len(rem)

    # expected (point-estimate) probabilities for reporting
    pred = fit.predict(rem["home"], rem["away"])

    draws = max(1, min(n_param_draws, n_sims))
    per = np.full(draws, n_sims // draws)
    per[: n_sims % draws] += 1
    thetas = fit.sample_theta(draws, rng) if parameter_uncertainty else np.repeat(fit.theta()[None], draws, 0)
    if drift_sd > 0 and R:
        share_left = R / (n * (n - 1))
        nt = len(fit.teams)
        thetas = thetas.copy()
        thetas[:, 3:] += rng.normal(0.0, drift_sd * np.sqrt(share_left), size=(draws, 2 * nt))

    ov_i, ov_l, ov_m = np.array([], int), np.array([]), np.array([])
    if overrides is not None and len(overrides) and R:
        key = {(h, a): k for k, (h, a) in enumerate(zip(rem["home"], rem["away"]))}
        trip = [(key[(h, a)], l, m) for h, a, l, m in
                zip(overrides["home"], overrides["away"], overrides["mkt_lh"], overrides["mkt_la"])
                if (h, a) in key and np.isfinite(l) and np.isfinite(m)]
        if trip:
            ov_i, ov_l, ov_m = (np.array(v) for v in zip(*trip))
            ov_i = ov_i.astype(int)

    G = 11
    fixed_rows = {}
    if fixed_dist and R:
        key = {(h, a): k for k, (h, a) in enumerate(zip(rem["home"], rem["away"]))}
        for (h, a), mat in fixed_dist.items():
            if (h, a) not in key:
                raise ValueError(f"{h} v {a} is not a remaining fixture")
            mat = np.asarray(mat, dtype=float)
            sq = np.zeros((G, G))
            r_, c_ = min(G, mat.shape[0]), min(G, mat.shape[1])
            sq[:r_, :c_] = mat[:r_, :c_]
            sq /= sq.sum()
            fixed_rows[key[(h, a)]] = sq.reshape(-1)

    hg_all = np.empty((n_sims, R), dtype=np.int16)
    ag_all = np.empty((n_sims, R), dtype=np.int16)
    s0 = 0
    for b in range(draws if R else 0):
        lam, mu = fit.rates(rem["home"], rem["away"], thetas[b])
        if len(ov_i):
            lam[ov_i] = ov_l
            mu[ov_i] = ov_m
        m = score_matrix(lam, mu, thetas[b][2], max_goals=G - 1).reshape(R, -1)
        for k_, row in fixed_rows.items():
            m[k_] = row
        cdf = np.cumsum(m, axis=1)
        cdf[:, -1] = 1.0
        u = rng.random((per[b], R))
        k = (cdf[None, :, :] < u[:, :, None]).sum(-1)
        hg_all[s0:s0 + per[b]] = k // G
        ag_all[s0:s0 + per[b]] = k % G
        s0 += per[b]

    # season totals
    win_h = hg_all > ag_all
    win_a = hg_all < ag_all
    draw = ~(win_h | win_a)
    pts = np.tile(pts0, (n_sims, 1)).astype(np.int32)
    gf = np.tile(gf0, (n_sims, 1)).astype(np.int32)
    ga = np.tile(ga0, (n_sims, 1)).astype(np.int32)
    hp = np.tile(hp0, (n_sims, 1, 1)).astype(np.int16)
    hgm = np.tile(hg0, (n_sims, 1, 1)).astype(np.int16)
    for r in range(R):
        i, j = rh[r], ra[r]
        x, y = hg_all[:, r], ag_all[:, r]
        p_i = 3 * win_h[:, r] + draw[:, r]
        p_j = 3 * win_a[:, r] + draw[:, r]
        pts[:, i] += p_i
        pts[:, j] += p_j
        gf[:, i] += x
        ga[:, i] += y
        gf[:, j] += y
        ga[:, j] += x
        hp[:, i, j] += p_i
        hp[:, j, i] += p_j
        hgm[:, i, j] += x
        hgm[:, j, i] += y
    gd = gf - ga

    full = np.full((n, n), 2)
    np.fill_diagonal(full, 0)
    positions = np.empty((n_sims, n), dtype=np.int16)
    fair_play = rng.random((n_sims, n))        # unpredictable last resort
    for s in range(n_sims):
        order = rank_order(pts[s], gd[s], gf[s], hp[s], hgm[s], full, fair_play[s])
        positions[s, order] = np.arange(1, n + 1)

    fixtures = pred.copy()
    return SimResult(teams=teams, positions=positions, points=pts, fixtures=fixtures, base_table=base)


In [ ]:
%%writefile zsc/evaluate.py
"""Scoring rules for probability forecasts (lower is better for all three)."""

from __future__ import annotations

import numpy as np


def outcome_index(hg, ag) -> np.ndarray:
    """0 = home win, 1 = draw, 2 = away win."""
    hg, ag = np.asarray(hg), np.asarray(ag)
    return np.where(hg > ag, 0, np.where(hg == ag, 1, 2))


def rps(probs: np.ndarray, outcome: np.ndarray) -> np.ndarray:
    """Ranked probability score per match (ordered outcomes H < D < A)."""
    probs = np.asarray(probs, float)
    obs = np.zeros_like(probs)
    obs[np.arange(len(outcome)), outcome] = 1.0
    cp = np.cumsum(probs, axis=1)[:, :2]
    co = np.cumsum(obs, axis=1)[:, :2]
    return ((cp - co) ** 2).sum(axis=1) / 2.0


def log_loss(probs: np.ndarray, outcome: np.ndarray) -> np.ndarray:
    p = np.clip(np.asarray(probs, float)[np.arange(len(outcome)), outcome], 1e-12, 1.0)
    return -np.log(p)


def brier(probs: np.ndarray, outcome: np.ndarray) -> np.ndarray:
    probs = np.asarray(probs, float)
    obs = np.zeros_like(probs)
    obs[np.arange(len(outcome)), outcome] = 1.0
    return ((probs - obs) ** 2).sum(axis=1)


def calibration_table(probs: np.ndarray, outcome: np.ndarray, bins=(0, .1, .2, .3, .4, .5, .6, .7, .8, .9, 1.0001)):
    """Rows: (bin low, bin high, n, mean forecast, observed frequency) over all 3 outcomes."""
    probs = np.asarray(probs, float)
    obs = np.zeros_like(probs)
    obs[np.arange(len(outcome)), outcome] = 1.0
    p, o = probs.ravel(), obs.ravel()
    rows = []
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (p >= lo) & (p < hi)
        if m.sum():
            rows.append((lo, min(hi, 1.0), int(m.sum()), float(p[m].mean()), float(o[m].mean())))
    return rows


In [ ]:
%%writefile zsc/pipeline.py
"""Glue: assemble the full match table used for fitting and forecasting."""

from __future__ import annotations

import os

import pandas as pd

from .data import attach_odds, load_history, load_unified_csv
from .market import add_market_goals

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))


def load_snapshot(root: str = ROOT, first_season: int = 2000) -> pd.DataFrame:
    """History (with Bet365 odds + shots on target) plus the current season."""
    hist = load_history(os.path.join(root, "data", "history"), first_season=first_season)
    odds = pd.read_csv(os.path.join(root, "data", "odds", "laliga_bet365_odds.csv"), parse_dates=["date"])
    hist = attach_odds(hist, odds)
    cur_path = os.path.join(root, "data", "current", "laliga_2026_27.csv")
    frames = [hist]
    if os.path.exists(cur_path):
        cur = load_unified_csv(cur_path)
        frames.append(cur[cur["season"] > hist["season"].max()])
    allm = pd.concat(frames, ignore_index=True).sort_values(["date", "home"]).reset_index(drop=True)
    return add_market_goals(allm)


def season_teams(matches: pd.DataFrame, season: int) -> list[str]:
    s = matches[matches["season"] == season]
    return sorted(set(s["home"]) | set(s["away"]))


def promoted_teams(matches: pd.DataFrame, season: int) -> set[str]:
    now = set(season_teams(matches, season))
    before = set(season_teams(matches, season - 1))
    return now - before if before else set()


In [ ]:
%%writefile zsc/forecast.py
"""End-to-end forecast for the current season."""

from __future__ import annotations

import json
import os
from dataclasses import asdict

import numpy as np
import pandas as pd

from .dixon_coles import DCConfig, fit_dixon_coles
from .pipeline import promoted_teams, season_teams
from .simulate import simulate_season
from .standings import league_table
from .teams import arabic, english


def load_settings(outputs_dir: str):
    """Tuned model settings (from scripts/tune.py and scripts/season_backtest.py)."""
    with open(os.path.join(outputs_dir, "best_config.json")) as f:
        cfg = DCConfig(**json.load(f))
    drift = 0.05
    sb = os.path.join(outputs_dir, "season_backtest.json")
    if os.path.exists(sb):
        with open(sb) as f:
            drift = float(json.load(f)["chosen_drift_sd"])
    return cfg, drift


def run_forecast(matches: pd.DataFrame, season: int, ref_date, cfg: DCConfig, drift_sd: float,
                 upcoming: pd.DataFrame | None = None, n_sims: int = 10_000, seed: int = 2026,
                 overrides: pd.DataFrame | None = None) -> dict:
    """overrides: optional (home, away, mkt_lh, mkt_la) market expected goals
    for the next fixtures, used instead of the model for those games."""
    ref_date = pd.Timestamp(ref_date)
    teams = season_teams(matches, season)
    played = matches[(matches["season"] == season) & (matches["date"] < ref_date)]
    fit = fit_dixon_coles(matches, ref_date, cfg, current_teams=teams,
                          promoted=promoted_teams(matches, season), current_season=season)
    # spread calibrated on 14 past seasons (see scripts/season_backtest.py)
    sim = simulate_season(fit, played, teams, n_sims=n_sims, drift_sd=drift_sd,
                          parameter_uncertainty=False, overrides=overrides, seed=seed)

    summary = sim.summary()
    table = league_table(played, teams)

    # team ratings on an easy scale: expected goal difference per game
    # against an average team on neutral ground
    idx = fit.index
    att_s, dfn_s = fit.scaled()
    ratings = pd.DataFrame({"team": teams})
    ratings["attack"] = [att_s[idx[t]] for t in teams]
    ratings["defence"] = [dfn_s[idx[t]] for t in teams]
    mean_a = ratings["attack"].mean()
    mean_d = ratings["defence"].mean()
    base = np.exp(fit.c + fit.h / 2)
    ratings["xg_for"] = base * np.exp(ratings["attack"] - mean_d)
    ratings["xg_against"] = base * np.exp(mean_a - ratings["defence"])
    ratings["xgd_per_game"] = ratings["xg_for"] - ratings["xg_against"]
    ratings = ratings.sort_values("xgd_per_game", ascending=False).reset_index(drop=True)

    out = {"fit": fit, "sim": sim, "summary": summary, "table": table, "ratings": ratings,
           "played": played, "config": cfg, "drift_sd": drift_sd, "ref_date": ref_date, "season": season}
    if upcoming is not None and len(upcoming):
        nxt = fit.predict(upcoming["home"], upcoming["away"])
        for c in ["date", "time_uk", "matchday"]:
            if c in upcoming:
                nxt.insert(0, c, upcoming[c].values)
        out["next_round"] = nxt
    return out


def to_json(res: dict, backtest: dict | None = None, season_bt: dict | None = None) -> dict:
    """Plain dict for the results page."""
    s = res["summary"]
    teams_out = []
    for _, r in s.iterrows():
        teams_out.append({
            "team": r["team"], "en": english(r["team"]), "ar": arabic(r["team"]),
            "pts_now": int(r["pts_now"]), "exp_pts": round(float(r["exp_pts"]), 1),
            "pts_p10": float(r["pts_p10"]), "pts_p90": float(r["pts_p90"]),
            "p_title": float(r["p_title"]), "p_top4": float(r["p_top4"]), "p_top6": float(r["p_top6"]),
            "p_releg": float(r["p_releg"]), "exp_pos": float(r["exp_pos"]),
            "pos": [float(r[f"pos_{k}"]) for k in range(1, len(s) + 1)],
        })
    table = []
    for _, r in res["table"].iterrows():
        table.append({"pos": int(r["Pos"]), "team": r["team"], "en": english(r["team"]), "ar": arabic(r["team"]),
                      "P": int(r["P"]), "W": int(r["W"]), "D": int(r["D"]), "L": int(r["L"]),
                      "GF": int(r["GF"]), "GA": int(r["GA"]), "GD": int(r["GD"]), "Pts": int(r["Pts"])})
    data = {
        "season": f"{res['season']}/{(res['season'] + 1) % 100:02d}",
        "as_of": res["ref_date"].strftime("%Y-%m-%d"),
        "matches_played": int(len(res["played"])),
        "n_sims": int(res["sim"].positions.shape[0]),
        "config": asdict(res["config"]),
        "drift_sd": res["drift_sd"],
        "teams": teams_out,
        "table": table,
        "ratings": [{"team": r["team"], "ar": arabic(r["team"]), "en": english(r["team"]),
                     "xg_for": round(float(r["xg_for"]), 2), "xg_against": round(float(r["xg_against"]), 2),
                     "xgd": round(float(r["xgd_per_game"]), 2)} for _, r in res["ratings"].iterrows()],
    }
    if "next_round" in res:
        data["next_round"] = [{
            "date": str(r.get("date", "")), "time_uk": str(r.get("time_uk", "")),
            "home": r["home"], "away": r["away"],
            "home_ar": arabic(r["home"]), "away_ar": arabic(r["away"]),
            "home_en": english(r["home"]), "away_en": english(r["away"]),
            "xg_home": round(float(r["xg_home"]), 2), "xg_away": round(float(r["xg_away"]), 2),
            "p_home": float(r["p_home"]), "p_draw": float(r["p_draw"]), "p_away": float(r["p_away"]),
            "likely_score": r["likely_score"], "p_likely_score": float(r["p_likely_score"]),
        } for _, r in res["next_round"].iterrows()]
    if backtest:
        data["backtest"] = backtest
    if season_bt:
        data["season_backtest"] = season_bt
    return data


In [ ]:
%%writefile zsc/online.py
"""Fresh data straight from the internet (used by the weekly Colab notebook).

Main source: football-data.co.uk (results, shots on target, Bet365 odds),
updated by the site twice a week. Fallback when that site is unreachable:
the datasets/football-datasets GitHub mirror for past seasons and
openfootball for the current season (results only, no odds).
"""

from __future__ import annotations

import io
import json
import urllib.request

import numpy as np
import pandas as pd

from .data import COLUMNS, fetch_football_data, season_code, standardise_football_data
from .market import add_market_goals, implied_1x2, implied_goals, implied_over
from .teams import canonical

MIRROR = "https://raw.githubusercontent.com/datasets/football-datasets/main/datasets/la-liga/season-{code}.csv"
OPENFOOTBALL = "https://raw.githubusercontent.com/openfootball/football.json/master/{y}-{y2}/es.1.json"
FIXTURES = "https://www.football-data.co.uk/fixtures.csv"


def _get(url: str, timeout: int = 60) -> bytes:
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (zsc model)"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()


def _openfootball_season(start_year: int) -> pd.DataFrame:
    js = json.loads(_get(OPENFOOTBALL.format(y=start_year, y2=f"{(start_year + 1) % 100:02d}")))
    rows = []
    for mt in js["matches"]:
        s = mt.get("score")
        ft = s.get("ft") if isinstance(s, dict) else (s if isinstance(s, list) else None)
        if not ft:
            continue
        rows.append({"date": pd.Timestamp(mt["date"]), "season": start_year,
                     "home": canonical(mt["team1"]), "away": canonical(mt["team2"]),
                     "hg": int(ft[0]), "ag": int(ft[1])})
    df = pd.DataFrame(rows)
    for c in COLUMNS:
        if c not in df:
            df[c] = np.nan
    return df[COLUMNS]


def merge_current(primary: pd.DataFrame, extra: pd.DataFrame, log=print) -> pd.DataFrame:
    """Add matches that only `extra` has (each home/away pair is played once a season).

    Rows present in both keep `primary` (it carries odds and shots). If the two
    sources disagree on any score, the second source is not used at all and a
    warning lists the disagreements.
    """
    if extra is None or extra.empty:
        return primary
    have = {(h, a): (x, y) for h, a, x, y in zip(primary["home"], primary["away"], primary["hg"], primary["ag"])}
    bad = [(h, a, have[(h, a)], (x, y)) for h, a, x, y in zip(extra["home"], extra["away"], extra["hg"], extra["ag"])
           if (h, a) in have and have[(h, a)] != (x, y)]
    if bad:
        log(f"  ⚠️ the second source disagrees on {len(bad)} score(s), so it was ignored: {bad}")
        return primary
    new = extra[[(h, a) not in have for h, a in zip(extra["home"], extra["away"])]]
    if len(new):
        log(f"  + {len(new)} newer results from the second source")
    return pd.concat([primary, new[primary.columns]], ignore_index=True)


def parse_extra_results(text: str, season: int) -> pd.DataFrame:
    """Lines 'YYYY-MM-DD, Home, Away, home goals, away goals' typed by hand."""
    rows = []
    for line in (text or "").strip().splitlines():
        if not line.strip() or line.strip().startswith("#"):
            continue
        d, h, a, x, y = [p.strip() for p in line.split(",")]
        rows.append({"date": pd.Timestamp(d), "season": season, "home": canonical(h), "away": canonical(a),
                     "hg": int(x), "ag": int(y)})
    df = pd.DataFrame(rows, columns=["date", "season", "home", "away", "hg", "ag"])
    for c in COLUMNS:
        if c not in df:
            df[c] = np.nan
    return df[COLUMNS]


def load_online(season: int, seasons_back: int = 4, extra_results: str = "", log=print) -> pd.DataFrame:
    frames = []
    for s in range(season - seasons_back, season + 1):
        try:
            df = fetch_football_data(s)
            log(f"  {s}/{(s + 1) % 100:02d}: {len(df)} matches from football-data.co.uk")
        except Exception as e:                        # noqa: BLE001
            log(f"  {s}/{(s + 1) % 100:02d}: football-data.co.uk failed ({e}); trying fallback")
            if s < season:
                raw = pd.read_csv(io.BytesIO(_get(MIRROR.format(code=season_code(s)))))
                df = standardise_football_data(raw, s)
            else:
                df = _openfootball_season(s)
            log(f"  {s}/{(s + 1) % 100:02d}: {len(df)} matches from fallback (no odds)")
        if s == season:
            # football-data.co.uk sometimes lags a few days: add anything newer from openfootball
            try:
                df = merge_current(df, _openfootball_season(s), log)
            except Exception as e:                    # noqa: BLE001
                log(f"  openfootball not reachable ({e})")
            manual = parse_extra_results(extra_results, s)
            if len(manual):
                df = merge_current(df, manual, log)
        frames.append(df)
    allm = pd.concat(frames, ignore_index=True).sort_values(["date", "home"]).reset_index(drop=True)
    return add_market_goals(allm)


def upcoming_fixtures(days: int = 8, today=None) -> pd.DataFrame:
    """Next fixtures from football-data.co.uk (with odds), else openfootball (no odds)."""
    try:
        up = upcoming_with_odds()
        if len(up):
            return up
    except Exception:                                 # noqa: BLE001
        pass
    today = pd.Timestamp.today().normalize() if today is None else pd.Timestamp(today)
    y = today.year if today.month >= 7 else today.year - 1
    js = json.loads(_get(OPENFOOTBALL.format(y=y, y2=f"{(y + 1) % 100:02d}")))
    rows = []
    for mt in js["matches"]:
        s = mt.get("score")
        ft = s.get("ft") if isinstance(s, dict) else (s if isinstance(s, list) else None)
        d = pd.Timestamp(mt["date"])
        if ft or not (today <= d <= today + pd.Timedelta(days=days)):
            continue
        rows.append({"date": d.strftime("%Y-%m-%d"), "time_uk": "", "home": canonical(mt["team1"]),
                     "away": canonical(mt["team2"]), "mkt_lh": np.nan, "mkt_la": np.nan})
    return pd.DataFrame(rows, columns=["date", "time_uk", "home", "away", "mkt_lh", "mkt_la"])


def upcoming_with_odds(days: int = 8) -> pd.DataFrame:
    """Next La Liga fixtures with Bet365 odds from football-data.co.uk/fixtures.csv."""
    raw = pd.read_csv(io.BytesIO(_get(FIXTURES)), encoding="utf-8-sig")
    raw = raw[raw["Div"] == "SP1"].copy()
    if raw.empty:
        return pd.DataFrame(columns=["date", "time_uk", "home", "away"])
    out = pd.DataFrame({
        "date": pd.to_datetime(raw["Date"], dayfirst=True).dt.strftime("%Y-%m-%d"),
        "time_uk": raw.get("Time", ""),
        "home": raw["HomeTeam"].map(canonical),
        "away": raw["AwayTeam"].map(canonical),
    })
    for a, b in [("odds_h", "B365H"), ("odds_d", "B365D"), ("odds_a", "B365A"),
                 ("odds_o25", "B365>2.5"), ("odds_u25", "B365<2.5")]:
        out[a] = pd.to_numeric(raw[b], errors="coerce") if b in raw else np.nan
    ok = out[["odds_h", "odds_d", "odds_a"]].notna().all(axis=1).values
    out["mkt_p_home"] = np.nan
    out["mkt_p_draw"] = np.nan
    out["mkt_p_away"] = np.nan
    out["mkt_lh"] = np.nan
    out["mkt_la"] = np.nan
    if ok.any():
        p = implied_1x2(out.loc[ok, "odds_h"], out.loc[ok, "odds_d"], out.loc[ok, "odds_a"])
        po = implied_over(out.loc[ok, "odds_o25"], out.loc[ok, "odds_u25"])
        lam, mu, _ = implied_goals(p[:, 0], p[:, 2], po)
        out.loc[ok, ["mkt_p_home", "mkt_p_draw", "mkt_p_away"]] = p
        out.loc[ok, "mkt_lh"] = lam
        out.loc[ok, "mkt_la"] = mu
    return out.reset_index(drop=True)


In [ ]:
%%writefile zsc/dashboard.py
"""Tables (CSV files) that feed the results dashboard."""

from __future__ import annotations

import os

import pandas as pd


def _pct(x: float) -> float:
    return round(100.0 * float(x), 1)


def uk_to_baghdad(date: str, time_uk: str) -> str:
    """'2026-10-25', '20:00' (UK local time) -> '23:00' (Baghdad). Empty if unknown."""
    from datetime import datetime
    from zoneinfo import ZoneInfo
    try:
        t = datetime.strptime(f"{date} {time_uk}", "%Y-%m-%d %H:%M").replace(tzinfo=ZoneInfo("Europe/London"))
        return t.astimezone(ZoneInfo("Asia/Baghdad")).strftime("%H:%M")
    except (ValueError, TypeError):
        return ""


def dashboard_tables(data: dict, accuracy: list[dict] | None = None) -> dict[str, pd.DataFrame]:
    """data = zsc.forecast.to_json(...) output."""
    out = {}
    nr = data.get("next_round", [])
    out["run_info"] = pd.DataFrame([{
        "season": data["season"], "as_of": data["as_of"], "matches_played": data["matches_played"],
        "n_sims": data["n_sims"], "n_sims_text": f"{data['n_sims']:,}", "next_matches": len(nr),
    }])
    out["forecast"] = pd.DataFrame([{
        "team": t["team"], "ar": t["ar"], "en": t["en"],
        "pts_now": t["pts_now"], "exp_pts": round(t["exp_pts"]),
        "pts_low": int(round(t["pts_p10"])), "pts_high": int(round(t["pts_p90"])),
        "title_pct": _pct(t["p_title"]), "top4_pct": _pct(t["p_top4"]),
        "europe_pct": _pct(t["p_top6"]), "releg_pct": _pct(t["p_releg"]),
        "exp_pos": round(t["exp_pos"], 1),
    } for t in data["teams"]])
    # title contenders first (the dashboard's three tiles read rows 0-2)
    out["forecast"] = out["forecast"].sort_values(["title_pct", "exp_pts"], ascending=False).reset_index(drop=True)
    out["positions"] = pd.DataFrame([{
        "team": t["team"], "ar": t["ar"], "pos": k + 1, "pct": _pct(p)}
        for t in data["teams"] for k, p in enumerate(t["pos"])])
    out["next_round"] = pd.DataFrame([{
        "match": f"{m['home']} v {m['away']}", "date": m["date"], "time_uk": m["time_uk"],
        "time_baghdad": uk_to_baghdad(m["date"], m["time_uk"]),
        "home": m["home"], "away": m["away"], "home_ar": m["home_ar"], "away_ar": m["away_ar"],
        "home_pct": _pct(m["p_home"]), "draw_pct": _pct(m["p_draw"]), "away_pct": _pct(m["p_away"]),
        "xg_home": m["xg_home"], "xg_away": m["xg_away"],
        "likely_score": m["likely_score"], "likely_score_pct": _pct(m["p_likely_score"]),
    } for m in nr])
    out["table_now"] = pd.DataFrame([{
        "pos": r["pos"], "team": r["team"], "ar": r["ar"], "P": r["P"], "W": r["W"], "D": r["D"],
        "L": r["L"], "GF": r["GF"], "GA": r["GA"], "GD": r["GD"], "Pts": r["Pts"]} for r in data["table"]])
    out["ratings"] = pd.DataFrame([{
        "team": r["team"], "ar": r["ar"], "xg_for": r["xg_for"], "xg_against": r["xg_against"],
        "xgd": r["xgd"]} for r in data["ratings"]])
    if accuracy:
        out["accuracy"] = pd.DataFrame(accuracy)
    return out


def accuracy_rows(backtest: dict, season_bt: dict, test_preds: pd.DataFrame) -> list[dict]:
    """Headline accuracy figures from scripts/tune.py and scripts/season_backtest.py.

    test_preds: outputs/backtest_predictions_test.csv (held-out matches)."""
    from .evaluate import outcome_index
    from .market import implied_1x2
    y = outcome_index(test_preds["hg"], test_preds["ag"])
    pm = test_preds[["p_home", "p_draw", "p_away"]].values
    pk = implied_1x2(test_preds["odds_h"], test_preds["odds_d"], test_preds["odds_a"])
    backtest = dict(backtest)
    backtest["hit_model"] = float((pm.argmax(1) == y).mean())
    backtest["hit_market"] = float((pk.argmax(1) == y).mean())
    backtest["hit_home"] = float((y == 0).mean())          # always pick the home team
    fm = backtest["final_model"]
    backtest["skill_share"] = 100 * (fm["rps_base_rates"] - fm["rps"]) / (fm["rps_base_rates"] - fm["rps_mkt"])
    sm = season_bt["metrics"]
    seasons = backtest["test_seasons"]
    period = f"{seasons[0]}/{(seasons[0] + 1) % 100:02d}-{seasons[-1]}/{(seasons[-1] + 1) % 100:02d}"
    rows = [
        {"id": "rps", "ar": "خطأ الاحتمالات (RPS) — كل ما أقل أحسن", "model": round(fm["rps"], 4),
         "bookmaker": round(fm["rps_mkt"], 4), "baseline": round(fm["rps_base_rates"], 4),
         "unit": "", "period": period, "n": fm["n"]},
        {"id": "hit", "ar": "نسبة إصابة النتيجة (فوز/تعادل/خسارة)", "model": round(100 * backtest["hit_model"], 1),
         "bookmaker": round(100 * backtest["hit_market"], 1), "baseline": round(100 * backtest["hit_home"], 1),
         "unit": "%", "period": period, "n": fm["n"]},
        {"id": "skill", "ar": "نسبة دقة الموديل من دقة شركات المراهنات", "model": round(backtest["skill_share"], 1),
         "bookmaker": 100.0, "baseline": 0.0, "unit": "%", "period": period, "n": fm["n"]},
        {"id": "points", "ar": "متوسط خطأ النقاط النهائية (توقع من الجولة 10)", "model": round(sm["points_MAE"], 1),
         "bookmaker": None, "baseline": round(sm["naive_points_MAE"], 1), "unit": "نقطة",
         "period": f"{season_bt['seasons'][0]}/{(season_bt['seasons'][0] + 1) % 100:02d}-"
                   f"{season_bt['seasons'][-1]}/{(season_bt['seasons'][-1] + 1) % 100:02d}",
         "n": sm["teams"]},
        {"id": "coverage", "ar": "مرات وقوع النقاط الحقيقية داخل مدى الـ80%", "model": round(100 * sm["coverage_80"], 1),
         "bookmaker": None, "baseline": 80.0, "unit": "%",
         "period": f"{season_bt['seasons'][0]}/{(season_bt['seasons'][0] + 1) % 100:02d}-"
                   f"{season_bt['seasons'][-1]}/{(season_bt['seasons'][-1] + 1) % 100:02d}",
         "n": sm["teams"]},
    ]
    return rows


def write_tables(tables: dict[str, pd.DataFrame], folder: str) -> list[str]:
    os.makedirs(folder, exist_ok=True)
    paths = []
    for name, df in tables.items():
        p = os.path.join(folder, f"{name}.csv")
        df.to_csv(p, index=False, encoding="utf-8")
        paths.append(p)
    return paths


## الإعدادات
هاي القيم طلعت من ضبط الموديل على 11 موسم واختباره على 3 مواسم ما شافها. لا تغيرينها إلا إذا تعرفين شتسوين.

In [ ]:
SEASON = 2026          # 2026 = موسم 2026/27
N_SIMS = 10_000        # عدد المواسم المحاكاة
CONFIG = {'xi': 0.006, 'prior_sd': 0.7, 'beta_sot': 0.0, 'beta_mkt': 0.9, 'promoted_attack': -0.19, 'promoted_defence': -0.13, 'seasons_back': 2, 'sot_k': None, 'rating_scale': 1.1}
DRIFT_SD = 0.05     # تغيّر مستوى الفرق خلال الموسم (مضبوط على التاريخ)
# دقة الموديل المقاسة على مواسم ما شافها (تنعرض بصفحة النتائج)
ACCURACY = [{'id': 'rps', 'ar': 'خطأ الاحتمالات (RPS) — كل ما أقل أحسن', 'model': 0.1924, 'bookmaker': 0.1893, 'baseline': 0.2253, 'unit': '', 'period': '2023/24-2025/26', 'n': 1140}, {'id': 'hit', 'ar': 'نسبة إصابة النتيجة (فوز/تعادل/خسارة)', 'model': 53.7, 'bookmaker': 55.2, 'baseline': 45.8, 'unit': '%', 'period': '2023/24-2025/26', 'n': 1140}, {'id': 'skill', 'ar': 'نسبة دقة الموديل من دقة شركات المراهنات', 'model': 91.4, 'bookmaker': 100.0, 'baseline': 0.0, 'unit': '%', 'period': '2023/24-2025/26', 'n': 1140}, {'id': 'points', 'ar': 'متوسط خطأ النقاط النهائية (توقع من الجولة 10)', 'model': 5.1, 'bookmaker': None, 'baseline': 9.2, 'unit': 'نقطة', 'period': '2012/13-2025/26', 'n': 280}, {'id': 'coverage', 'ar': 'مرات وقوع النقاط الحقيقية داخل مدى الـ80%', 'model': 82.9, 'bookmaker': None, 'baseline': 80.0, 'unit': '%', 'period': '2012/13-2025/26', 'n': 280}]


## 1) سحب البيانات والتحقق منها
إذا آخر مباراة بالبيانات أقدم من آخر جولة (المواقع أحياناً تتأخر يوم أو يومين)، اكتبي النتائج الناقصة بالمربع `EXTRA_RESULTS` بهذا الشكل، كل مباراة بسطر:

`2026-10-25, Barcelona, Real Madrid, 2, 1`

(التاريخ، صاحب الأرض، الضيف، أهداف الأرض، أهداف الضيف). الأسماء بالإنكليزي مثل جدول football-data.

In [ ]:
EXTRA_RESULTS = '''
'''

import pandas as pd, numpy as np
from zsc.online import load_online, upcoming_fixtures
from zsc.data import validate_season
from zsc.dixon_coles import DCConfig
from zsc.forecast import run_forecast, to_json
from zsc.teams import arabic
pd.set_option('display.width', 200)

cfg = DCConfig(**CONFIG)
matches = load_online(SEASON, seasons_back=max(cfg.seasons_back, 2), extra_results=EXTRA_RESULTS)
cur = matches[matches.season == SEASON]
problems = validate_season(cur, n_teams=20)
assert not problems, problems
print(f'مباريات الموسم الحالي: {len(cur)} — آخر مباراة: {cur.date.max().date()}')
print(f'مباريات بيها أودز: {cur.odds_h.notna().sum()} من {len(cur)}')
print('✅ البيانات سليمة: 20 فريق، بدون تكرار')

In [ ]:
try:
    upcoming = upcoming_fixtures()
    print(f'مباريات الجولة الجاية: {len(upcoming)} (منها {upcoming.mkt_lh.notna().sum()} بيها أودز)')
except Exception as e:
    upcoming = pd.DataFrame(columns=['date','time_uk','home','away','mkt_lh','mkt_la'])
    print('ما كدرت أجيب مباريات الجولة الجاية:', e)
# keep only fixtures still to be played
done = set(zip(cur.home, cur.away))
upcoming = upcoming[[(h, a) not in done for h, a in zip(upcoming.home, upcoming.away)]].reset_index(drop=True)
upcoming

## 2) تشغيل الحاسوب الخارق

In [ ]:
# use every match played up to now
ref_date = cur.date.max() + pd.Timedelta(days=1)
# for the very next matches the betting market's view is used (it knows team news)
res = run_forecast(matches, SEASON, ref_date, cfg, DRIFT_SD,
                   upcoming=upcoming if len(upcoming) else None, n_sims=N_SIMS,
                   overrides=upcoming if len(upcoming) else None)
print(f'✅ خلصت المحاكاة: {N_SIMS:,} موسم — البيانات لغاية {cur.date.max().date()}')

## 3) النتائج

In [ ]:
s = res['summary'].copy()
show = pd.DataFrame({
    'الفريق': s.team.map(arabic),
    'نقاطه الآن': s.pts_now,
    'النقاط المتوقعة': s.exp_pts.round(1),
    'اللقب %': (100 * s.p_title).round(1),
    'أول 4 %': (100 * s.p_top4).round(1),
    'أوروبا (أول 6) %': (100 * s.p_top6).round(1),
    'الهبوط %': (100 * s.p_releg).round(1),
})
show.index = range(1, len(show) + 1)
show

In [ ]:
if 'next_round' in res:
    n = res['next_round'].copy()
    tab = pd.DataFrame({
        'المباراة': n.home.map(arabic) + ' × ' + n.away.map(arabic),
        'فوز الأرض %': (100 * n.p_home).round(0), 'تعادل %': (100 * n.p_draw).round(0),
        'فوز الضيف %': (100 * n.p_away).round(0),
        'الأهداف المتوقعة': n.xg_home.round(2).astype(str) + ' - ' + n.xg_away.round(2).astype(str),
        'النتيجة الأرجح': n.likely_score,
    })
    if 'mkt_p_home' in upcoming:
        tab['السوق: أرض/تعادل/ضيف %'] = [f"{a:.0f}/{b:.0f}/{c:.0f}" if pd.notna(a) else '-' for a, b, c in
                                       zip(100*upcoming.mkt_p_home, 100*upcoming.mkt_p_draw, 100*upcoming.mkt_p_away)]
    display(tab)

In [ ]:
import matplotlib.pyplot as plt
import arabic_reshaper
from bidi.algorithm import get_display
ar = lambda t: get_display(arabic_reshaper.reshape(t))
top = s[s.p_title > 0.001].sort_values('p_title')
fig, ax = plt.subplots(figsize=(8, 0.5 * len(top) + 1.5))
ax.barh([ar(arabic(t)) for t in top.team], 100 * top.p_title, color='#a50044')
for y, v in enumerate(100 * top.p_title):
    ax.text(v + 0.5, y, f'{v:.1f}%', va='center')
ax.set_title(ar('فرصة الفوز باللقب — حاسوب زهراء الخارق'))
ax.set_xlim(0, 105); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout(); plt.show()

## 4) حفظ الملفات
`dashboard_files.zip` هو الملف اللي تبعثيه حتى تتحدث صفحة النتائج. و`forecast_table.csv` جدول تفتحينه بـExcel.

In [ ]:
import json, shutil
from zsc.dashboard import dashboard_tables, write_tables
data = to_json(res)
with open('forecast.json', 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=1)
write_tables(dashboard_tables(data, ACCURACY), 'dashboard_files')
shutil.make_archive('dashboard_files', 'zip', 'dashboard_files')
show.to_csv('forecast_table.csv', encoding='utf-8-sig')
try:
    from google.colab import files
    files.download('dashboard_files.zip'); files.download('forecast_table.csv')
except Exception:
    print('الملفات محفوظة بنفس المجلد: dashboard_files.zip و forecast_table.csv')